In [4]:
library(rsample)     # data splitting 
library(dplyr)       # data wrangling
library(rpart)       # performing regression trees
library(janitor)
library(readr)
library(tidyverse)
library(xgboost)
library(rPref)
library(caret)
library(geometry)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘janitor’


The following objects are masked from ‘package:stats’:

    chisq.test, fisher.test


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘rPref’


The following object is masked from ‘package:dplyr’:

    between


Loading required package: lattice


Attaching package: ‘caret’


The following object is masked fro

### Constants

In [5]:
def_outbreak <- 2       # number of cases to be considered and outbreak
perc_strata <- 0.75  # training/test split

### Creating data with no filtering or SMOTE

In [6]:
set.seed(100)
# Choose which dataset to use for testing 
#df <- read_csv("data/merged_data.csv", show_col_types = FALSE) |> clean_names() 
df <- read_csv("data/merged_with_svi.csv", show_col_types = FALSE) |> clean_names()

# SVI specific, because it does not include the data for Mcculloch, Mclennan, Mcmullen and Dewitt, 
# if we continue the prev way of data processing it will remove all SVI related features, so we remove these 4 counties.
df <- df[!df$county %in% c("Mcculloch", "Mclennan", "Mcmullen", "Dewitt"), ]

# Remove redundant columns
df_raw <- df
df$density <- df$population * 1.0 / df$area_sqmi
df <- df %>% select(-c("county", "phr", "area_sqmi"),
  -starts_with("m_"), -starts_with("mp_"), -starts_with("e_"), -starts_with("epl_"),
  -starts_with("spl_"), -starts_with("rpl_"), -starts_with("f_"))
df <- df %>% select(-c("ep_minrty", "ep_hisp", "ep_afam", "ep_pov150", "ep_uninsur"))

# split the data with stratified sampling
#strata <- ifelse(df$outbreak > 0, "nonzero", "zero")
index <- createDataPartition(df$outbreak, p = perc_strata, list = FALSE)
train <- df[index,]
test <- df[-index,]

train <- train[, !names(train) %in% c("county")]
test <- test[, !names(test) %in% c("county")]
train$outbreak <- as.integer(train$outbreak >= def_outbreak)
test$outbreak <- as.integer(test$outbreak >= def_outbreak)


### Xgboost Model

In [7]:
set.seed(100)
cutoff = 0.5

X_train <- as.matrix(train[, names(train) != "outbreak"])
y_train <- as.numeric(as.character(train$outbreak))

X_test  <- as.matrix(test[, names(test) != "outbreak"])
y_test  <- as.numeric(as.character(test$outbreak))

dtrain  <- xgb.DMatrix(data = X_train, label = y_train)
dtest   <- xgb.DMatrix(data = X_test,  label = y_test)

boost_model <- xgb.train(
  params = list(
    objective   = "binary:logistic",
    eval_metric = "logloss"
  ),
  data    = dtrain,
  nrounds = 500
)

pred_xgb <- as.integer(predict(boost_model, dtest)>= cutoff)

boost_table <- table(Actual = y_test, Predicted = pred_xgb)
boost_table

# importance graph
importance_matrix <- xgb.importance(model = boost_model)
#xgb.plot.importance(importance_matrix)
#importance_matrix[order(importance_matrix$Gain, decreasing = TRUE)]

# Summary of first tree model
# tree_df <- xgb.model.dt.tree(model = boost_model)
# print(tree_df[Tree == 0, ])

TP <- boost_table[2, 2] # True Positives (Actual Outbreak, Predicted Outbreak)
TN <- boost_table[1, 1] # True Negatives (Actual No Outbreak, Predicted No Outbreak)
FP <- boost_table[1, 2] # False Positives (Actual No Outbreak, Predicted Outbreak)
FN <- boost_table[2, 1] # False Negatives (Actual Outbreak, Predicted No Outbreak)

accuracy    <- (TP + TN) / sum(boost_table)
recall <- TP / (TP + FN) 
precision   <- TP / (TP + FP)

cat("Model Performance Summary (Cutoff = 0.5):\n",
    "Accuracy (Overall Correctness): ", round(accuracy, 4), "\n",
    "Recall (Outbreak Capture Rate): ", round(recall, 4), "\n",
    "Precision (Reliability of Outbreak Prediction): ", round(precision, 4), "\n")

      Predicted
Actual  0  1
     0 52  2
     1  8  0

Model Performance Summary (Cutoff = 0.5):
 Accuracy (Overall Correctness):  0.8387 
 Recall (Outbreak Capture Rate):  0 
 Precision (Reliability of Outbreak Prediction):  0 


### Parameter Tuning

In [8]:
set.seed(100)
# Xgboost 
c_xgb <- c(seq(0.2, 0.5, by=0.1))      # deciding what counts as outbreak
max_depth = c(seq(3, 9, by=2))         # max depth of each tree 
eta = c(0.01,seq(0.05, 0.4, by=0.05))  # learning rate

# Create grid
param_grid <- expand.grid(
  max_depth = max_depth,
  eta = eta,
  cutoff = c_xgb
)

# Create folds for testing
folds <- createFolds(train$outbreak, k=10)

### Cross validation w/ parameter tuning

In [9]:
set.seed(100)
results_xgb <- data.frame()

for(i in 1:nrow(param_grid)){
  fold_precision <- c()
  fold_recall    <- c()
  fold_accuracy  <- c()
    
  for(fold in folds){
    fold_train <- train[-fold, ]
    fold_test  <- train[fold, ]

    X_fold_train <- as.matrix(fold_train[, names(fold_train) != "outbreak"])
    y_fold_train <- as.numeric(as.character(fold_train$outbreak))
    X_fold_test  <- as.matrix(fold_test[, names(fold_test) != "outbreak"])
    y_fold_test  <- as.numeric(as.character(fold_test$outbreak))

    dtrain_fold <- xgb.DMatrix(data = X_fold_train, label = y_fold_train)
    dtest_fold  <- xgb.DMatrix(data = X_fold_test,  label = y_fold_test)

    model <- xgb.train(
      params = list(
        objective   = "binary:logistic",
        eval_metric = "logloss",
        max_depth   = param_grid$max_depth[i],
        eta         = param_grid$eta[i]
      ),
      data    = dtrain_fold,
      nrounds = 500,
      verbose = 0
    )

    preds <- as.numeric(predict(model, dtest_fold) >= param_grid$cutoff[i])

    # build full 2x2 confusion matrix even if a class is missing
    cm <- table(
      factor(fold_test$outbreak, levels = c(0, 1)),
      factor(preds,              levels = c(0, 1))
    )

    tp <- cm[2, 2]
    fp <- cm[1, 2]
    fn <- cm[2, 1]

    precision <- if((tp + fp) == 0) NA else tp / (tp + fp)
    recall    <- if((tp + fn) == 0) NA else tp / (tp + fn)
    accuracy  <- sum(diag(cm)) / sum(cm)

    fold_precision <- c(fold_precision, precision)
    fold_recall    <- c(fold_recall,    recall)
    fold_accuracy  <- c(fold_accuracy,  accuracy)
  }

  # outside inner loop
  results_xgb <- rbind(results_xgb, data.frame(
    max_depth     = param_grid$max_depth[i],
    eta           = param_grid$eta[i],
    cutoff        = param_grid$cutoff[i],
    precision     = mean(fold_precision, na.rm = TRUE),
    precision_std = sd(fold_precision,   na.rm = TRUE),
    recall        = mean(fold_recall,    na.rm = TRUE),
    recall_std    = sd(fold_recall,      na.rm = TRUE),
    accuracy      = mean(fold_accuracy,  na.rm = TRUE),
    accuracy_std  = sd(fold_accuracy,    na.rm = TRUE)
  ))
}

best_xgb <- results_xgb[which.max(results_xgb$recall), ]
print(best_xgb)

  max_depth  eta cutoff precision precision_std   recall recall_std accuracy
1         3 0.01    0.2 0.4166667     0.3632416 0.462963  0.4697648 0.904386
  accuracy_std
1   0.05426758


### Evaluation best parameters

In [10]:
p_xgb <- high(precision, df = results_xgb) * high(recall) * high(accuracy)
res_xgb <- psel(results_xgb, p_xgb)

# Create the actual convex hull for the full results vector
hull_mat <- as.matrix(results_xgb[, c("precision", "recall", "accuracy")])
hull_idx <- convhulln(hull_mat) 

# hull_idx is a dataframe of triangles (surface) that encapsulate all points
# find only the unique values so they all connect
unique_indices <- unique(as.vector(hull_idx))
results_hull <- results_xgb[unique_indices, ]

# Take only intersecting points of the data
intersecting_points <- merge(results_hull, res_xgb)
res_xgb <- intersecting_points[order(intersecting_points$precision), ]
res_xgb

,max_depth,eta,cutoff,precision,precision_std,recall,recall_std,accuracy,accuracy_std
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,3,0.01,0.2,0.4166667,0.3632416,0.4629630,0.4697648,0.9043860,0.05426758
2,5,0.01,0.4,0.6666667,0.4714045,0.2962963,0.4547418,0.9307018,0.05612630


### Evaluating Optimized parameter metrics

In [11]:
set.seed(100)

# Setting optimal parameters
max_depth = 3
eta = 0.4
cutoff = 0.4

# Create matricies for xgboost 
X_train <- as.matrix(train[, names(train) != "outbreak"])
y_train <- as.numeric(as.character(train$outbreak))

X_test  <- as.matrix(test[, names(test) != "outbreak"])
y_test  <- as.numeric(as.character(test$outbreak))

dtrain  <- xgb.DMatrix(data = X_train, label = y_train)
dtest   <- xgb.DMatrix(data = X_test,  label = y_test)


# Train model with parameters
final_model <- xgb.train(
  params = list(
    objective   = "binary:logistic",
    eval_metric = "logloss",
    eta = eta, 
    max_depth = max_depth
  ),
  data    = dtrain,
  nrounds = 500
)

# Predict on test set and display results
pred_xgb <- as.integer(predict(final_model, dtest)>= cutoff)
boost_table <- table(Actual = y_test, Predicted = pred_xgb)
boost_table

# Calculate Precision, Accuracy, and recall
TP <- boost_table[2, 2] # True Positives 
TN <- boost_table[1, 1] # True Negatives 
FP <- boost_table[1, 2] # False Positives 
FN <- boost_table[2, 1] # False Negatives 

accuracy    <- (TP + TN) / sum(boost_table) # total accuracy
recall <- TP / (TP + FN)                    # amount of positives guessed correctly out of all true positives
precision   <- TP / (TP + FP)               # amount of correct positive predictions

cat("Model Performance Summary:\n",
    "Accuracy (Overall Correctness): ", round(accuracy, 4), "\n",
    "Recall (Outbreak Capture Rate): ", round(recall, 4), "\n",
    "Precision (Reliability of Outbreak Prediction): ", round(precision, 4), "\n")


#         Predicted
# Actual  0  1
#      0 48  9
#      1  1  4


      Predicted
Actual  0  1
     0 52  2
     1  7  1

Model Performance Summary:
 Accuracy (Overall Correctness):  0.8548 
 Recall (Outbreak Capture Rate):  0.125 
 Precision (Reliability of Outbreak Prediction):  0.3333 


### Investigating False Results

In [12]:
# Looking at if false negatives is consistent with other classification methods
false_negatives <- test[test$outbreak == 1 & pred_xgb == 0, ]
false_negatives # IS the same county (upshur)


cve,outbreak,enrollment,population,pct_hispanic,pct_black,pct_white,pct_poverty,pct_uninsured,pct_college,⋯,ep_crowd,ep_noveh,ep_groupq,ep_noint,ep_asian,ep_aian,ep_nhpi,ep_twomore,ep_otherrace,density
<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
0.80,1,1330,7057,66.0,0.9,71.3,7.4,28.6,15.2,⋯,5.4,13.2,2.7,19.3,1.3,1.0,0.0,1.4,0.0,8.533547
2.06,1,2071,7831,54.3,0.2,50.9,10.7,24.4,11.9,⋯,3.9,3.4,0.6,18.5,0.0,0.0,0.0,3.3,1.5,5.209789
3.99,1,2515,18223,17.3,2.2,80.8,10.5,15.3,21.8,⋯,2.7,6.3,5.5,18.7,0.4,0.4,0.0,1.9,0.2,19.668483
1.60,1,35221,175677,63.5,4.5,48.1,14.0,22.1,17.6,⋯,6.6,5.2,1.2,15.7,1.0,0.3,0.0,1.6,0.1,195.656788
3.24,1,8516,12620,9.6,12.1,75.1,15.7,19.3,21.6,⋯,2.1,7.1,1.5,23.3,0.9,0.7,0.2,4.3,0.1,13.908931
1.07,1,2164,9288,66.7,0.3,64.0,8.9,24.1,16.9,⋯,4.9,1.9,0.8,14.8,0.0,0.3,0.0,1.5,0.2,10.544703
5.00,1,7416,44174,10.4,7.2,79.3,10.1,13.8,18.2,⋯,2.4,3.9,1.2,13.0,0.5,0.1,0.1,3.3,0.2,75.772807


### Looking at splits for optimzied tree

In [13]:
# Extract all trees into df 
tree_summary <- xgb.model.dt.tree(model = final_model)

# look at first tree
first_tree <- tree_summary[Tree == 5]
first_tree

#false_negatives[, c("any_cancer_no", "spl_theme2", "e_unemp", "cve")]

# SVI documentation: https://www.atsdr.cdc.gov/place-health/media/pdfs/2024/10/SVI2022Documentation.pdf
# any_cancer_no: does not have cancer
# enrollment: percentage of kinds enrolled in school
# m_hh: house holds estimate margin of error? 
# ep_aian: Percentage of American Indian or Alaska Native, not Hispanic or Latino persons estimate, 

# Guide on how to read text dump: 
# look at a given feature (x) then split number (y)
# all formulas will be in the form x < y 
# given this look at the yes or no column the right value is the node that it will traverse to next

# Gain (for non-leaf nodes): "importance level" arbitrary value to determine value 
# Gain (for leaf nodes): probability increase/decrease of outbreak (sigmoid log odds of it happening)
# Cover: the average number of observations effected by this feature to split 

Tree,Node,ID,Feature,Split,Yes,No,Missing,Gain,Cover
<int>,<int>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>
5,0,5-0,cve,1.88,5-1,5-2,5-2,1.57160521,8.487002
5,1,5-1,routine_checkup_within_the_past_5_years,6.60,5-3,5-4,5-3,0.02989459,2.526760
5,2,5-2,ep_nohsdp,14.30,5-5,5-6,5-6,2.26937556,5.960242
5,3,5-3,Leaf,NA,NA,NA,NA,-0.29580471,1.513718
5,4,5-4,Leaf,NA,NA,NA,NA,-0.07443503,1.013042
5,5,5-5,Leaf,NA,NA,NA,NA,-0.21277948,1.971099
5,6,5-6,ep_hburd,20.00,5-7,5-8,5-8,1.38521588,3.989143
5,7,5-7,Leaf,NA,NA,NA,NA,-0.03087975,1.814647
5,8,5-8,Leaf,NA,NA,NA,NA,0.39168793,2.174496


### Looking into m_hh (house hold estimate margin or error)

In [14]:
# Household estimate is the amount of housing units that are in a given population 
# We try to look into the variance of the this variable
# I just realized that this i the first 

# Look at margin of error
summary(df$m_hh)
#hist(df$m_hh, breaks = 100)

# Closer look into MOE 
hist(df$m_hh, breaks = 100, xlim = c(0, 1000))

# Look at actual household estimate 
summary(df$e_hh)
# hist(df$e_hh)

Warning message:
“Unknown or uninitialised column: `m_hh`.”


Length  Class   Mode 
     0   NULL   NULL 

Warning message:
“Unknown or uninitialised column: `m_hh`.”


ERROR: Error in hist.default(df$m_hh, breaks = 100, xlim = c(0, 1000)): 'x' must be numeric


### Look at all features used in the model

In [15]:
# importance graph
# Gain: the average imporvement in the models accuracy when the feature is used to split 
# Cover: The average number of observations effected by this feature to split 
# Frequency: The percentage of times a feature is chosen to split the data accross all trees

importance_matrix <- xgb.importance(model = boost_model)
#head(importance_matrix[order(importance_matrix$Gain, decreasing = TRUE)], 10)
importance_matrix

# Print all features used in the boosted tree
# print(importance_matrix[order(importance_matrix$Gain, decreasing = TRUE)])

# List of most important features: 
# Any_cancer_no: has cancer - greater people with cancer highly correlated 
# Enrollment: not sure 
# ep_age17: amount of people age 17 or under 
# spl_theme1: sum of socioeconomic status indicators
# ep_limeng: amount of limited english speakers
# m_limeng: margin of error ^
# sigm_and_blood_still_50_70_i_no: unsure 
# e_munit: multi unit housing 
# cve: concious exemption rate 
# ep_aian: Native american percentage 
# e_groupq: Persons in group quarter

Feature,Gain,Cover,Frequency
<chr>,<dbl>,<dbl>,<dbl>
any_cancer_no,2.370389e-01,0.057471028,0.022935780
ep_age65,1.907333e-01,0.108154963,0.091743119
pct_black,1.194476e-01,0.052831763,0.045871560
cve,9.161283e-02,0.100462615,0.091743119
ep_unemp,6.805565e-02,0.058911677,0.045871560
mammogram_no,5.217315e-02,0.038232343,0.013761468
ep_disabl,4.563647e-02,0.003040356,0.004587156
ep_nohsdp,4.139025e-02,0.034801077,0.036697248
pct_hispanic,3.499640e-02,0.081677630,0.064220183


### Look at intersecting features from removed na data and no removed na data

In [18]:
yes_NA_features <- importance_matrix$Feature

write.csv(importance_matrix, "data/results/important_vars_xgboost_w_NA.csv")

# Notice most of the important features are used in the models, at least for the yes NA features model
# intersected_features <- intersect(no_NA_features, yes_NA_features)
# intersected_features